# Ближайшие соседи и гибрид с DNN

Ближайшие соседи ищутся по среднему косинусному сходству эмбеддингов GeoCLIP и StreetCLIP. Модель взвешивает координаты соседей по этому сходству и уточняет среднее направление на единичной сфере ECEF. В гибриде направление смешивается с прогнозом DNN сферической интерполяцией, а веса выбираются по тем же пяти фолдам с оценкой на WGS84.

Сначала запустите [подготовку данных](00-preparation.ipynb). Выбор конфигурации основан на среднем квадрате геодезической ошибки по пяти фолдам.

В приватном репозитории GitHub выберите **Code → Download ZIP**, загрузите архив в Colab и укажите `ZIP_PATH`, например `/content/train250k-geo-ml-main.zip`. Если репозиторий уже клонирован, укажите `REPO_DIR` или запустите Jupyter из корня проекта. `REPO_URL` можно использовать, если доступ Git уже настроен.

По умолчанию результаты хранятся в `/content/train250k-cv`. Если будете переключать среды выполнения, используйте один и тот же `WORK_DIR` на Google Диске во всех блокнотах, например `/content/drive/MyDrive/train250k-cv`.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPO_URL = ""  # @param {type:"string"}
ZIP_PATH = ""  # @param {type:"string"}
REPO_DIR = "/content/train250k-geo-ml"  # @param {type:"string"}
WORK_DIR = "/content/train250k-cv"  # @param {type:"string"}
INSTALL_DEPENDENCIES = True  # @param {type:"boolean"}

checkout = Path(REPO_DIR).expanduser()
local_candidates = (checkout, Path.cwd(), Path.cwd().parent)
repo = next((path for path in local_candidates if (path / "src" / "experiment.py").is_file()), None)
if repo is None and ZIP_PATH:
    from tempfile import TemporaryDirectory
    from zipfile import ZipFile

    if checkout.exists():
        raise FileExistsError("Выберите свободный путь REPO_DIR для распаковки.")
    checkout.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(dir=checkout.parent) as tmp:
        with ZipFile(Path(ZIP_PATH).expanduser()) as archive:
            archive.extractall(tmp)
        extracted = next((path for path in Path(tmp).iterdir() if (path / "src" / "experiment.py").is_file()), None)
        if extracted is None:
            raise FileNotFoundError("В архиве GitHub отсутствует src/experiment.py.")
        extracted.rename(checkout)
    repo = checkout
if repo is None:
    if not REPO_URL:
        raise ValueError("Укажите ZIP_PATH, REPO_DIR к готовому репозиторию или REPO_URL с настроенным доступом Git.")
    subprocess.run(["git", "clone", REPO_URL, str(checkout)], check=True)
    repo = checkout

work = Path(WORK_DIR).expanduser().resolve()
env = {**os.environ, "GEO_CV_ROOT": str(work)}
if work.is_relative_to("/content/drive") and not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")

if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")], check=True)


def run_src(script, *arguments):
    subprocess.run([sys.executable, "-u", str(repo / "src" / script), *arguments], cwd=repo, env=env, check=True)

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Выберите GPU в настройках среды выполнения Colab и перезапустите ячейки.")
if not (work / "split.json").is_file():
    raise FileNotFoundError("Запустите 00-preparation.ipynb с тем же WORK_DIR.")
print(torch.cuda.get_device_name())

Для гибрида нужны валидационные прогнозы [DNN](02-dnn.ipynb).

In [ ]:
if not (work / "cv-dnn-summary.json").is_file():
    raise FileNotFoundError("Сначала запустите 02-dnn.ipynb с тем же WORK_DIR.")

In [ ]:
run_src("experiment.py", "neighbors")

## Результаты кросс-валидации

In [ ]:
import pandas as pd
from IPython.display import display

summary = pd.read_json(work / "cv-neighbors-summary.json")
table = summary[["config", "mae_km", "rmse_km", "mse_km2", "n"]].rename(columns={
    "config": "Конфигурация", "mae_km": "MAE, км", "rmse_km": "RMSE, км",
    "mse_km2": "MSE, км²", "n": "Точек",
})
display(table)